In [1]:
import numpy as np
import pandas as pd
import ast
import random
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

I0000 00:00:1778773578.788027  237637 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778773579.202817  237637 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778773581.172741  237637 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
seed_value = 10
np.random.seed(seed_value)
random.seed(seed_value)
tf.random.set_seed(seed_value)

# 2. DATA EXTRACTION FUNCTION
def extract_L_set(filepath):
    l_set = []
    capture = False
    with open(filepath, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("L set:"):
                capture = True
                continue
            if line.startswith("B vector:"):
                capture = False
                break
            if capture and line.startswith("["):
                row = ast.literal_eval(line)
                l_set.append(row)
    return np.array(l_set)

In [3]:
def get_direction_acc(y_true, y_pred):
    # Convert to 1 (Rising) or -1 (Falling)
    true_dir = np.where(np.array(y_true) >= 0, 1, -1)
    pred_dir = np.where(np.array(y_pred) >= 0, 1, -1)
    matches = (true_dir == pred_dir)
    return np.mean(matches) * 100


In [4]:
train_val_path = '../ED_Calculation/2003_2023/results/finalize_L_set_from_2003_to_2023.txt'
test_path = "../ED_Calculation/2024_current/results/final_result_df_2024_2025.txt"

data_2003_2023 = extract_L_set(train_val_path)
data_2024_2025 = extract_L_set(test_path)

# Split 2003-2023 into Val (first 30) and Train (rest)
X_val_raw = data_2003_2023[:30, :5]
y_val = data_2003_2023[:30, 5]

X_train_raw = data_2003_2023[30:, :5]
y_train = data_2003_2023[30:, 5]

X_test_raw = data_2024_2025[:, :5]
y_test = data_2024_2025[:, 5]

# Scaling (Standardize based on training data)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

In [11]:
# --- PART A: KNN IMPLEMENTATION ---
print("--- Optimizing KNN ---")
best_k = 1
best_knn_val_acc = 0

# Test different K values on the 30 validation samples
for k in range(1, 25, 1):
    knn = KNeighborsRegressor(n_neighbors=k, metric='euclidean')
    knn.fit(X_train, y_train)
    val_preds = knn.predict(X_val)
    acc = get_direction_acc(y_val, val_preds)
    
    if acc > best_knn_val_acc:
        best_knn_val_acc = acc
        best_k = k
    print(f"KNN K: {k} (Val Acc: {best_knn_val_acc}%)")

print(f"Best KNN K: {best_k} (Val Acc: {best_knn_val_acc}%)")

--- Optimizing KNN ---
KNN K: 1 (Val Acc: 46.666666666666664%)
KNN K: 2 (Val Acc: 46.666666666666664%)
KNN K: 3 (Val Acc: 46.666666666666664%)
KNN K: 4 (Val Acc: 46.666666666666664%)
KNN K: 5 (Val Acc: 46.666666666666664%)
KNN K: 6 (Val Acc: 46.666666666666664%)
KNN K: 7 (Val Acc: 46.666666666666664%)
KNN K: 8 (Val Acc: 46.666666666666664%)
KNN K: 9 (Val Acc: 46.666666666666664%)
KNN K: 10 (Val Acc: 46.666666666666664%)
KNN K: 11 (Val Acc: 46.666666666666664%)
KNN K: 12 (Val Acc: 46.666666666666664%)
KNN K: 13 (Val Acc: 46.666666666666664%)
KNN K: 14 (Val Acc: 46.666666666666664%)
KNN K: 15 (Val Acc: 46.666666666666664%)
KNN K: 16 (Val Acc: 46.666666666666664%)
KNN K: 17 (Val Acc: 46.666666666666664%)
KNN K: 18 (Val Acc: 46.666666666666664%)
KNN K: 19 (Val Acc: 46.666666666666664%)
KNN K: 20 (Val Acc: 46.666666666666664%)
KNN K: 21 (Val Acc: 46.666666666666664%)
KNN K: 22 (Val Acc: 46.666666666666664%)
KNN K: 23 (Val Acc: 46.666666666666664%)
KNN K: 24 (Val Acc: 46.666666666666664%)
Be

In [ ]:
# Final KNN Model
final_knn = KNeighborsRegressor(n_neighbors=best_k, metric='euclidean', weights='distance')
final_knn.fit(X_train, y_train)


In [6]:


# --- PART B: FFNN IMPLEMENTATION (From your PDF) ---
print("\n--- Training FFNN ---")
model = Sequential([
    Dense(16, activation='relu', input_shape=(5,)),
    Dense(1, activation='linear')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

model.fit(X_train, y_train, 
          validation_data=(X_val, y_val), 
          epochs=30, 
          batch_size=8, 
          callbacks=[early_stop], 
          verbose=0)

# --- PART C: ENSEMBLE INTEGRATION ---
print("\n--- Final Results (2024-2025 Test Set) ---")

# Get raw predictions
knn_test_preds = final_knn.predict(X_test)
ffnn_test_preds = model.predict(X_test).flatten()

# Individual Accuracies
knn_acc = get_direction_acc(y_test, knn_test_preds)
ffnn_acc = get_direction_acc(y_test, ffnn_test_preds)

# Ensemble: Average the predictions
ensemble_preds = (knn_test_preds + ffnn_test_preds) / 2
ensemble_acc = get_direction_acc(y_test, ensemble_preds)

# Consensus: Only trade if they agree
matches = 0
total_agreements = 0
for i in range(len(y_test)):
    k_dir = 1 if knn_test_preds[i] >= 0 else -1
    f_dir = 1 if ffnn_test_preds[i] >= 0 else -1
    actual = 1 if y_test[i] >= 0 else -1
    
    if k_dir == f_dir:
        total_agreements += 1
        if k_dir == actual:
            matches += 1

consensus_acc = (matches / total_agreements * 100) if total_agreements > 0 else 0

print(f"KNN Accuracy: {knn_acc:.2f}%")
print(f"FFNN Accuracy: {ffnn_acc:.2f}%")
print(f"Ensemble (Mean) Accuracy: {ensemble_acc:.2f}%")
print(f"Consensus Accuracy (When they agree): {consensus_acc:.2f}%")
print(f"Confidence Rate: {total_agreements}/{len(y_test)} samples")


--- Training FFNN ---


/home/sria/git-project/artificial-neural-network-for-time-series/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1778773582.991382  237637 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)



--- Final Results (2024-2025 Test Set) ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
KNN Accuracy: 45.45%
FFNN Accuracy: 63.64%
Ensemble (Mean) Accuracy: 54.55%
Consensus Accuracy (When they agree): 57.14%
Confidence Rate: 7/11 samples


In [7]:
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# 1. OPTIMIZE KNN ON VALIDATION SET
# We use the raw scaled data to find what history 'suggests'
best_k = 5 # Based on previous runs, k=5 is a stable daily start
knn_feat_gen = KNeighborsRegressor(n_neighbors=best_k, weights='distance')
knn_feat_gen.fit(X_train, y_train)

# 2. GENERATE THE "HISTORICAL SUGGESTION" FEATURE
# We create a new column that represents the KNN's prediction
train_knn_feat = knn_feat_gen.predict(X_train).reshape(-1, 1)
val_knn_feat = knn_feat_gen.predict(X_val).reshape(-1, 1)
test_knn_feat = knn_feat_gen.predict(X_test).reshape(-1, 1)

# 3. APPEND THIS AS THE 6TH COLUMN
X_train_augmented = np.hstack((X_train, train_knn_feat))
X_val_augmented = np.hstack((X_val, val_knn_feat))
X_test_augmented = np.hstack((X_test, test_knn_feat))

# 4. BUILD NEW FFNN (Input shape is now 6)
model_aug = Sequential([
    Dense(16, activation='relu', input_shape=(6,)), # Note the '6' here
    Dense(8, activation='relu'), # Adding a small extra layer for complexity
    Dense(1, activation='linear')
])

model_aug.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

# 5. TRAIN
model_aug.fit(X_train_augmented, y_train, 
              validation_data=(X_val_augmented, y_val),
              epochs=100, batch_size=8, verbose=0)

# 6. EVALUATE
aug_preds = model_aug.predict(X_test_augmented).flatten()
final_acc = get_direction_acc(y_test, aug_preds)

print(f"Augmented FFNN Accuracy: {final_acc:.2f}%")

/home/sria/git-project/artificial-neural-network-for-time-series/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Augmented FFNN Accuracy: 36.36%


In [8]:
from sklearn.model_selection import cross_val_predict

# 1. OPTIMIZE KNN ON VALIDATION SET
best_k = 5 
knn_feat_gen = KNeighborsRegressor(n_neighbors=best_k, weights='distance')

# 2. GENERATE THE FEATURE WITHOUT "LEAKAGE" (The Fix)
# cross_val_predict ensures the KNN never predicts on data it was trained on
train_knn_feat = cross_val_predict(knn_feat_gen, X_train, y_train, cv=5).reshape(-1, 1)

# Now fit the KNN on all training data to predict Val and Test
knn_feat_gen.fit(X_train, y_train)
val_knn_feat = knn_feat_gen.predict(X_val).reshape(-1, 1)
test_knn_feat = knn_feat_gen.predict(X_test).reshape(-1, 1)

# SCALE THE NEW FEATURE (Crucial so it doesn't overpower the neural network)
from sklearn.preprocessing import StandardScaler
feat_scaler = StandardScaler()
train_knn_feat = feat_scaler.fit_transform(train_knn_feat)
val_knn_feat = feat_scaler.transform(val_knn_feat)
test_knn_feat = feat_scaler.transform(test_knn_feat)

# 3. APPEND THIS AS THE 6TH COLUMN
X_train_augmented = np.hstack((X_train, train_knn_feat))
X_val_augmented = np.hstack((X_val, val_knn_feat))
X_test_augmented = np.hstack((X_test, test_knn_feat))